In [35]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd
LAST_NAME = "Celo"
STUDENT_ID = "TUPM-26-1941"
id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT
np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)


Student: Celo TUPM-26-1941
Seed Digit: 1
ID Sum: 23
Name Length: 4
Random Seed: 261941
Verification Value: 93


In [36]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")

Workspace: C:\Users\admin\Lab7_NumPy_Pandas_Celo
All dataset files verified.


In [37]:
# Load the SECOM measurement matrix and identify features containing missing or constant observed values.
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
assert raw_measurements.ndim == 2

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))


In [38]:
rng = np.random.default_rng(RANDOM_SEED)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

def sensitivity_pipeline(value):
    data = np.random.default_rng(value) 

    indices = []
    for idx in usable_indices:
        column = raw_measurements[:, idx]
        p_count = (~np.isnan(column[labels == -1])).sum()
        f_count = (~np.isnan(column[labels == 1])).sum()
        if p_count>= 10 and f_count>=10:
            indices.append(idx)

    pick = data.choice(indices, size=5, replace=False)
    p_name = [f"Feature_{i:03d}" for i in pick]

# 4. Implement stratified bootstrap resampling using 200 + 10 x SEED_DIGIT repetitions, sampling with replacement independently within each outcome group.
    boot = 200 + 10 * SEED_DIGIT
    record = []

    for idx, feature in zip(pick, p_name):
        c_raw = raw_measurements[:, idx]
# 2.  Implement two preparation strategies: available-case analysis for each feature and median imputation using the full dataset.
        c_median = np.nanmedian(c_raw)
        c_imputed = np.where(np.isnan(c_raw), c_median, c_raw)

# Available-Case Analysis
        aca_pass = c_raw[(labels == -1) & (~np.isnan(c_raw))]
        aca_fail = c_raw[(labels == 1) & (~np.isnan(c_raw))]
        aca_difference = aca_fail.mean() - aca_pass.mean()

# Median Imputation
        mi_pass = c_imputed[labels == -1]
        mi_fail = c_imputed[labels == 1]
        mi_difference = mi_fail.mean() - mi_pass.mean()

# 3. For each strategy, calculate the difference between Fail and Pass group means. 
        b_aca_difference = np.empty(boot)
        b_mi_difference = np.empty(boot)

        for b in range(boot):
            s_apass = data.choice(aca_pass, size=len(aca_pass), replace=True)
            s_afail = data.choice(aca_fail, size=len(aca_fail), replace=True)
            b_aca_difference[b] = s_afail.mean() - s_apass.mean()

            s_mpass = data.choice(mi_pass, size=len(mi_pass), replace=True)
            s_mfail = data.choice(mi_fail, size=len(mi_fail), replace=True)
            b_mi_difference[b] = s_mfail.mean() - s_mpass.mean()

# 5. Calculate the bootstrap mean difference and percentile-based 95% confidence interval for each selected feature and strategy.
        ci_a_low, ci_a_high = np.percentile(b_aca_difference, [2.5, 97.5])
        ci_m_low, ci_m_high = np.percentile(b_mi_difference, [2.5, 97.5])

        record.append({
            "Feature": feature,
            "Available-Case Analysis Mean Difference": aca_difference,
            "Median Imputation Mean Difference": mi_difference,
            "Absolute Mean Change": abs(mi_difference - aca_difference),
            "Available-Case Analysis Bootstrap Mean": b_aca_difference.mean(),
            "Available-Case Analysis Low Confidence Interval": ci_a_low,
            "Available-Case Analysis High Confidence Interval": ci_a_high,
            "Median Imputation Bootstrap Mean": b_mi_difference.mean(),
            "Median Imputation Low Confidence Interval": ci_m_low,
            "Median Imputation High Confidence Interval": ci_m_high
        })
        
    results = pd.DataFrame(record)
    results["Available-Case Analysis Rank"] = results["Available-Case Analysis Mean Difference"].abs().rank(ascending=False).astype(int)
    results["Median Imputation Rank"] = results["Median Imputation Mean Difference"].abs().rank(ascending=False).astype(int)
    results["Sign Changed"] = np.sign(results["Available-Case Analysis Mean Difference"]) != np.sign(results["Median Imputation Mean Difference"])
    results["Rank Changed"] = results["Available-Case Analysis Rank"] != results["Median Imputation Rank"]
    results["Available-Case Analysis Confidence Interval Excluding Zero"] = (results["Available-Case Analysis Low Confidence Interval"] > 0) | (results["Available-Case Analysis High Confidence Interval"] < 0)
    results["Median Imputation Confidence Interval Excluding Zero"] = (results["Median Imputation Low Confidence Interval"] > 0) | (results["Median Imputation High Confidence Interval"] < 0)

    return results, boot

# 7. Verify reproducibility using the same random seed, then export the statistical comparison and a brief interpretation report.
robust, b_reproducibility= sensitivity_pipeline(RANDOM_SEED)
verifying, _ = sensitivity_pipeline(RANDOM_SEED)
reproducible = robust.equals(verifying)
robust.to_csv("robustness_comparison.csv", index=False)

c_sign = robust.loc[robust["Sign Changed"], "Feature"].tolist()
c_rank = robust.loc[robust["Rank Changed"], "Feature"].tolist()

report = (
    "SENSITIVITY AND ROBUSTNESS ANALYSIS REPORT\n"
    f"Student: {LAST_NAME} ({STUDENT_ID})\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Bootstrap Repetitions: {b_reproducibility}\n"
    f"Selected Features: {', '.join(robust['Feature'])}\n"
    f"Largest Change in Mean Difference: {robust['Absolute Mean Change'].max():.4f}\n"
    f"Features with Changed Difference Sign: {', '.join(c_sign) if c_sign else 'None'}\n"
    f"Features with Changed Ranking: {', '.join(c_rank) if c_rank else 'None'}\n"
    f"Reproducibility Verified: {reproducible}\n"
)

In [39]:
# [Assessment Data] Output Parameter

Path("sensitivity_report.txt").write_text(report, encoding="utf-8")
print("Selected Features:", ", ".join(robust["Feature"]))
print("Bootstrap Repetitions:", b_reproducibility)
print(f"Largest Change in Mean Difference: {robust['Absolute Mean Change'].max():.4f}")
print("Features with Changed Difference Sign:", ", ".join(c_sign) if c_sign else "None")
print("Features with Changed Ranking:", ", ".join(c_rank) if c_rank else "None")
print("Reproducibility Verified:", reproducible)


Selected Features: Feature_142, Feature_211, Feature_573, Feature_385, Feature_224
Bootstrap Repetitions: 210
Largest Change in Mean Difference: 0.0038
Features with Changed Difference Sign: None
Features with Changed Ranking: Feature_211, Feature_224
Reproducibility Verified: True
